In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Step 1: Configure 4-bit quantization
# Together, these 4 settings implement the QLoRA paper configuration, which is the most common setup for fine-tuning large models on consumer GPUs.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
# Step 2: Load tokenizer and model
model_name = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
# Step 3: Set to evaluation mode (CRITICAL!)
model.eval()

In [ ]:
# Step 4: Prepare input
prompt = "Explain the concept of Self-Attention in Transformers."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
# Step 5: Run inference (CRITICAL: disable Autograd)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )


In [ ]:
# Step 6: Decode and print response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)